In [1]:
import pandas as pd

df = pd.read_csv('../Processed_Headings/normalized_section_data 2.csv')
# df['sec-title'] = df.apply(lambda row: row['xmlPath'].rsplit(' > ', 1)[-1].strip(), axis=1)
# df.to_csv('Processed_Headings/combined.csv')
df.head()

,PMCID,PMID,sec-type,sec-title,sec-norm
0,PMC5427117,28337662,NaN,Introduction,background
1,PMC5427117,28337662,NaN,Methods,methods
2,PMC5427117,28337662,NaN,Results,results
3,PMC5427117,28337662,NaN,Discussion,conclusions
4,PMC5427117,28337662,NaN,Electronic supplementary material,background


In [3]:
import pandas as pd

df_test = pd.read_csv("../data/train_annotated.csv")
df_test

,PMCID,PMID,sec-title,sec-norm
0,PMC11191691,38842535,"['Introduction', 'Results and Discussion', 'Co...","['background', 'conclusions', 'conclusions', '..."
1,PMC11182025,38887617,"['Introduction', 'Materials & Methods', 'Resul...","['background', 'methods', 'results', 'conclusi..."
2,PMC10511328,37704722,"['Main', 'Pooled longitudinal analyses', 'Popu...","['background', 'results', 'results', 'results'..."
3,PMC10632145,37914938,"['Main', 'Population imaging and single-cell a...","['background', 'results', 'results', 'results'..."
4,PMC1084334,15884974,"['Introduction', 'Results', 'Discussion', 'Mat...","['background', 'results', 'conclusions', 'meth..."
...,...,...,...,...
635,PMC11187391,38123960,['Ethics statements'],[nan]
636,PMC11187998,38903349,"['Introduction', 'Case presentation', 'Discuss...","['background', 'methods', 'conclusions', 'conc..."
637,PMC11191920,38905421,"['1. Introduction', '2. Patients and methods',...","['background', 'methods', 'results', 'conclusi..."
638,PMC10978839,38548747,"['Introduction', 'Results', 'Discussion', 'Mat...","['background', 'results', 'conclusions', 'meth..."


In [4]:
import ast
df_test['sec-title'] = df_test['sec-title'].apply(ast.literal_eval)
df_test['sec-title']

0      [Introduction, Results and Discussion, Conclus...
1      [Introduction, Materials & Methods, Results, D...
2      [Main, Pooled longitudinal analyses, Populatio...
3      [Main, Population imaging and single-cell acti...
4      [Introduction, Results, Discussion, Materials ...
                             ...                        
635                                  [Ethics statements]
636    [Introduction, Case presentation, Discussion, ...
637    [1. Introduction, 2. Patients and methods, 3. ...
638    [Introduction, Results, Discussion, Materials ...
639    [Introduction, Construction of Chiral Nanomate...
Name: sec-title, Length: 640, dtype: object

In [5]:
import numpy as np
def eval_with_nan(x):
    if isinstance(x, list):
        return [np.nan if (isinstance(i, float) and np.isnan(i)) else i for i in x]
    try:
        # 這邊直接 eval 而且給定 locals
        val = eval(x, {"nan": np.nan, "NaN": np.nan, "np": np})
        if isinstance(val, list):
            return [np.nan if (isinstance(i, float) and np.isnan(i)) else i for i in val]
        else:
            return val
    except Exception:
        return x  # 如果 eval 還失敗就原樣保留


In [6]:
df_test['sec-norm'] = df_test['sec-norm'].apply(eval_with_nan)
df_test['sec-norm']

0      [background, conclusions, conclusions, backgro...
1      [background, methods, results, conclusions, co...
2      [background, results, results, results, result...
3      [background, results, results, results, result...
4      [background, results, conclusions, methods, ba...
                             ...                        
635                                                [nan]
636      [background, methods, conclusions, conclusions]
637    [background, methods, results, conclusions, ba...
638    [background, results, conclusions, methods, ba...
639    [background, background, background, backgroun...
Name: sec-norm, Length: 640, dtype: object

In [7]:
type(df_test['sec-norm'][0])

list

In [8]:
from itertools import chain
texts = list(chain.from_iterable(df_test["sec-title"]))
texts

['Introduction',
 'Results and Discussion',
 'Conclusions',
 'Supplementary Material',
 'Introduction',
 'Materials & Methods',
 'Results',
 'Discussion',
 'Conclusions',
 'Supplemental Information',
 'Additional Information and Declarations',
 'Main',
 'Pooled longitudinal analyses',
 'Population intervention effects',
 'Age-varying effects on growth faltering',
 'Consequences of early growth faltering',
 'Discussion',
 'Methods',
 'Online content',
 'Supplementary information',
 'Extended data',
 'Main',
 'Population imaging and single-cell activation',
 'Signal propagation map',
 'Functional measurements differ from anatomy',
 'Extrasynaptic signalling also drives neural dynamics',
 'Extrasynaptic-dependent signal propagation\xa0screen',
 'Signal propagation predicts spontaneous activity',
 'Discussion',
 'Methods',
 'Online content',
 'Supplementary information',
 'Source data',
 'Extended data',
 'Introduction',
 'Results',
 'Discussion',
 'Materials and Methods',
 'Supporting Inf

In [9]:

from sentence_transformers import SentenceTransformer, InputExample, losses
fine_tuned_model = SentenceTransformer("all-MiniLM-L6-v2")

# 測試新模型
embeddings = fine_tuned_model.encode([text.lower() for text in texts])

In [10]:
print(fine_tuned_model.tokenizer.model_max_length)

256


In [12]:
token_lens = []

for sent in texts:
    tokens = fine_tuned_model.tokenizer(
        sent,
        truncation=False,           # 不要截斷，讓我們看到原始長度
        return_attention_mask=False,
        return_token_type_ids=False
    )
    token_lens.append(len(tokens["input_ids"]))

import numpy as np
print(f"🔍 Token length stats:")
print(f"  Max: {max(token_lens)}")
print(f"  Mean: {np.mean(token_lens):.2f}")
print(f"  95th percentile: {np.percentile(token_lens, 95):.2f}")


🔍 Token length stats:
  Max: 44
  Mean: 4.53
  95th percentile: 9.00


In [13]:
import numpy as np

np.save("comparison/embeddings_test_lower0.npy", embeddings)

In [14]:
norms = list(chain.from_iterable(df_test["sec-norm"]))
norms

['background',
 'conclusions',
 'conclusions',
 'background',
 'background',
 'methods',
 'results',
 'conclusions',
 'conclusions',
 'background',
 'background',
 'background',
 'results',
 'results',
 'results',
 'results',
 'conclusions',
 'methods',
 'background',
 'background',
 'background',
 'background',
 'results',
 'results',
 'results',
 'results',
 'results',
 'results',
 'conclusions',
 'methods',
 'background',
 'background',
 'background',
 'background',
 'background',
 'results',
 'conclusions',
 'methods',
 'background',
 'background',
 'methods',
 'results',
 'conclusions',
 'background',
 'methods',
 'results',
 'conclusions',
 'conclusions',
 'background',
 'objective',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'methods',
 'conclusions',
 'conclusions',
 'background',
 'background',
 'background',
 'background',
 'background',
 'results',
 'conclusions',
 'methods',
 'backgrou

In [15]:
label2id = {
     "methods" : 0,
     "background" : 1,
     "results" : 2,
     "conclusions" : 3, 
     "objective": 4
}
labels = np.array([label2id[label] if label in label2id else -1 for label in norms])
np.save("comparison/labels_lower0.npy", labels)